**PRÁCTICA 5 · COMPARATIVA DE MODELOS (CLÁSICOS, CNN Y DEEPFACE)**

Este cuaderno carga todos los modelos entrenados y los evalúa **en las mismas condiciones** sobre el mismo conjunto de test, para comparar:

- **Resultados**: accuracy, accuracy balanceada, F1 macro y F1 por clase.
- **Rapidez de inferencia**: latencia de **una cara** (lo que importa en la webcam), incluyendo la extracción de características. Se mide la mediana sobre 200 caras.
- **Coste**: tamaño del modelo en disco y tiempo de entrenamiento (de `informe_entrenamiento/registro_modelos.json`, que escribe cada cuaderno de entrenamiento).
- **Frente a DeepFace** (`DeepFace.analyze`): sobre un subconjunto equilibrado del test sin `duckface` (DeepFace no tiene esa clase; su `disgust` se cuenta como `angry`).

Los modelos que se entrenaron con el dataset antiguo (8 clases) se omiten con un aviso: hay que volver a ejecutar su cuaderno.

In [ ]:
import json, os, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score)

from comun import EMOCIONES, cargar_dataset, preparar_cnn

N = len(EMOCIONES)
IDX_DUCK = EMOCIONES.index("duckface")
DIR = Path("informe_entrenamiento"); DIR.mkdir(exist_ok=True)
N_LATENCIA = 200          # caras para medir la latencia individual
N_DEEPFACE = 100          # caras por clase para la comparación con DeepFace
SEED = 42

MODELOS = [
    ("HOG + SVM", "modelos/hog_svm.joblib"),
    ("LBP + SVM", "modelos/lbp_svm.joblib"),
    ("HOG+LBP + PCA + SVM", "modelos/hoglbp_pca_svm.joblib"),
    ("Autocaras (PCA) + SVM", "modelos/autocaras_svm.joblib"),
    ("HOG + PCA + KNN", "modelos/hog_pca_knn.joblib"),
    ("CNN mini-Xception", "modelos/cnn.keras"),
]

imgs_te, yte = cargar_dataset("dataset/test")          # caras originales 48x48; cada modelo redimensiona
rng = np.random.default_rng(SEED)
idx_lat = rng.choice(len(imgs_te), N_LATENCIA, replace=False)
registro = json.loads((DIR / "registro_modelos.json").read_text(encoding="utf-8")) \
    if (DIR / "registro_modelos.json").exists() else {}

**EVALUACIÓN DE LOS MODELOS PROPIOS**

- Clásicos: el `.joblib` es un pipeline completo (extractor + escalado + PCA + clasificador), así que recibe directamente la cara en gris. La latencia incluye la extracción de HOG/LBP.
- CNN: se carga `cnn.keras`, se preprocesa con `preparar_cnn` y la latencia se mide con `tf.function`, igual que en la webcam. Se añade también la fila con TTA (promedio con la imagen volteada).

In [ ]:
def metricas(y, pred):
    k = y != IDX_DUCK
    return {"accuracy": accuracy_score(y, pred),
            "balanced_acc": balanced_accuracy_score(y, pred),
            "f1_macro": f1_score(y, pred, average="macro", labels=range(N), zero_division=0),
            "acc_sin_duckface": accuracy_score(y[k], pred[k])}


def latencia(fn):
    fn(imgs_te[idx_lat[0]])                               # calentamiento
    t = []
    for i in idx_lat:
        t0 = time.perf_counter(); fn(imgs_te[i]); t.append(time.perf_counter() - t0)
    return 1000 * float(np.median(t))


filas, predicciones, f1_clase = [], {}, {}
for nombre, ruta in MODELOS:
    if not os.path.exists(ruta):
        print(f"[omitido] {nombre}: no existe {ruta} (ejecuta su cuaderno de entrenamiento)")
        continue
    t0 = time.time()
    if ruta.endswith(".keras"):
        import keras, tensorflow as tf
        red = keras.models.load_model(ruta)
        t_carga = time.time() - t0
        X = np.array([preparar_cnn(i) for i in imgs_te])
        t0 = time.time(); p = red.predict(X, batch_size=512, verbose=0); t_lote = time.time() - t0
        pred = p.argmax(1)
        f = tf.function(lambda x: red(x, training=False))
        ms = latencia(lambda img: f(tf.constant(preparar_cnn(img)[None])).numpy())
        p_tta = 0.5 * (p + red.predict(X[:, :, ::-1, :], batch_size=512, verbose=0))
        extra = {"parametros": red.count_params()}
        predicciones[nombre + " + TTA"] = p_tta.argmax(1)
    else:
        m = joblib.load(ruta)
        t_carga = time.time() - t0
        if len(m.classes_) != N:
            print(f"[omitido] {nombre}: tiene {len(m.classes_)} clases (dataset antiguo). Vuelve a entrenarlo.")
            continue
        t0 = time.time(); pred = m.predict(imgs_te); t_lote = time.time() - t0
        ms = latencia(lambda img: m.predict([img]))
        extra = {}
    predicciones[nombre] = pred
    reg = registro.get(os.path.basename(ruta), {})
    if not reg and ruta.endswith(".keras") and (DIR / "resumen.json").exists():   # la CNN lo guarda también aquí
        reg = {"t_entreno_s": json.loads((DIR / "resumen.json").read_text(encoding="utf-8"))["entrenamiento"]["tiempo_total_s"]}
    filas.append({"modelo": nombre, **metricas(yte, pred), "ms_por_cara": ms,
                  "ms_por_imagen_en_lote": 1000 * t_lote / len(yte),
                  "tamaño_MB": os.path.getsize(ruta) / 2**20, "carga_s": t_carga,
                  "entrenamiento_s": reg.get("t_entreno_s"), **extra})
    f1_clase[nombre] = f1_score(yte, pred, average=None, labels=range(N), zero_division=0)
    print(f"{nombre:24s} acc {filas[-1]['accuracy']:.4f} | F1 {filas[-1]['f1_macro']:.4f} | {ms:.2f} ms/cara")
    if ruta.endswith(".keras"):
        filas.append({"modelo": nombre + " + TTA", **metricas(yte, predicciones[nombre + " + TTA"]),
                      "ms_por_cara": 2 * ms, "tamaño_MB": filas[-1]["tamaño_MB"],
                      "entrenamiento_s": reg.get("t_entreno_s")})

tabla = pd.DataFrame(filas).set_index("modelo")
tabla.to_csv(DIR / "comparativa_modelos.csv")
display(tabla.round(4))

**F1 POR CLASE**

Permite ver si un modelo gana en general o solo en algunas clases (por ejemplo, los clásicos suelen ir bien en `happy`, `surprise` y `duckface`, que tienen rasgos geométricos muy marcados, y mal en `fear` y `sad`).

In [ ]:
tabla_f1 = pd.DataFrame(f1_clase, index=EMOCIONES).T
tabla_f1.to_csv(DIR / "comparativa_f1_por_clase.csv")
display(tabla_f1.round(3))

**COMPARACIÓN CON DEEPFACE**

Se toma un subconjunto equilibrado de `N_DEEPFACE` caras por cada una de las 6 emociones que DeepFace conoce y se ejecuta `DeepFace.analyze(actions=["emotion"], detector_backend="skip")`, es decir, sin volver a detectar la cara (ya viene recortada). Sobre **las mismas caras** se calcula la accuracy de nuestros modelos.

Dos matices al interpretar el resultado:
- El modelo de emociones de DeepFace está **entrenado con FER-2013** y espera justo caras de 48x48 en gris, así que juega en casa; no podemos garantizar que alguna de estas imágenes no formara parte de su entrenamiento.
- Nuestros modelos pueden responder `duckface`, que aquí siempre es un error; DeepFace no tiene esa opción.

In [ ]:
from deepface import DeepFace

MAPA_DF = {"angry": "angry", "disgust": "angry", "fear": "fear", "happy": "happy",
           "sad": "sad", "surprise": "surprise", "neutral": "neutral"}
idx_df = np.concatenate([rng.choice(np.flatnonzero(yte == c), min(N_DEEPFACE, (yte == c).sum()), replace=False)
                         for c in range(N) if c != IDX_DUCK])

DeepFace.analyze(np.zeros((48, 48, 3), np.uint8), actions=["emotion"], detector_backend="skip",
                 enforce_detection=False, silent=True)
pred_df, tiempos = [], []
for i in idx_df:
    t0 = time.perf_counter()
    r = DeepFace.analyze(cv2.cvtColor(imgs_te[i], cv2.COLOR_GRAY2BGR), actions=["emotion"],
                         detector_backend="skip", enforce_detection=False, silent=True)[0]["emotion"]
    tiempos.append(time.perf_counter() - t0)
    p = np.zeros(N)
    for k, v in r.items():
        p[EMOCIONES.index(MAPA_DF[k])] += v
    pred_df.append(int(p.argmax()))
pred_df = np.array(pred_df)

y_sub = yte[idx_df]
filas_df = [{"modelo": "DeepFace.analyze", "accuracy": accuracy_score(y_sub, pred_df),
             "f1_macro": f1_score(y_sub, pred_df, average="macro", labels=[c for c in range(N) if c != IDX_DUCK], zero_division=0),
             "ms_por_cara": 1000 * float(np.median(tiempos))}]
for nombre, pred in predicciones.items():
    filas_df.append({"modelo": nombre, "accuracy": accuracy_score(y_sub, pred[idx_df]),
                     "f1_macro": f1_score(y_sub, pred[idx_df], average="macro",
                                          labels=[c for c in range(N) if c != IDX_DUCK], zero_division=0),
                     "ms_por_cara": tabla.loc[nombre, "ms_por_cara"]})
tabla_df = pd.DataFrame(filas_df).set_index("modelo").sort_values("accuracy", ascending=False)
tabla_df.to_csv(DIR / "comparativa_deepface.csv")
print(f"Subconjunto: {len(idx_df)} caras ({N_DEEPFACE} por emoción, sin duckface)")
display(tabla_df.round(4))

**GRÁFICOS RESUMEN**

1. F1 macro en test de cada modelo propio.
2. Compromiso calidad / velocidad: F1 macro frente a la latencia por cara (escala log), con el tamaño del punto proporcional al tamaño del modelo en disco.
3. Matrices de confusión normalizadas (recall por fila) de cada modelo.

In [ ]:
t = tabla.sort_values("f1_macro")
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].barh(t.index, t["f1_macro"], color="#4a7fb5")
ax[0].barh(t.index, t["accuracy"], fill=False, edgecolor="k", lw=1, label="accuracy")
for i, v in enumerate(t["f1_macro"]):
    ax[0].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)
ax[0].set_xlabel("F1 macro (relleno) / accuracy (contorno)"); ax[0].set_xlim(0, 1); ax[0].legend(loc="lower right")
ax[0].set_title("Calidad en test")

s = tabla.dropna(subset=["ms_por_cara"])
ax[1].scatter(s["ms_por_cara"], s["f1_macro"], s=20 + 3 * np.sqrt(s["tamaño_MB"].fillna(1)) * 10, alpha=0.6, color="#c0504d")
for n, r in s.iterrows():
    ax[1].annotate(f"{n}\n{r['tamaño_MB']:.1f} MB", (r["ms_por_cara"], r["f1_macro"]), fontsize=7,
                   xytext=(5, 5), textcoords="offset points")
ax[1].set_xscale("log"); ax[1].set_xlabel("latencia por cara (ms, escala log)"); ax[1].set_ylabel("F1 macro")
ax[1].set_title("Calidad frente a velocidad (tamaño del punto = tamaño del modelo)"); ax[1].grid(alpha=.3)
plt.tight_layout(); fig.savefig(DIR / "comparativa_modelos.png", dpi=150); plt.show()

n = len(predicciones)
fig, axs = plt.subplots(1, n, figsize=(3.3 * n, 3.6))
for a, (nombre, pred) in zip(np.atleast_1d(axs), predicciones.items()):
    cm = confusion_matrix(yte, pred, labels=range(N)).astype(float)
    cm /= cm.sum(1, keepdims=True).clip(min=1)
    a.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    a.set_title(nombre, fontsize=9)
    a.set_xticks(range(N)); a.set_xticklabels(EMOCIONES, rotation=90, fontsize=7)
    a.set_yticks(range(N)); a.set_yticklabels(EMOCIONES, fontsize=7)
    for i in range(N):
        a.text(i, i, f"{cm[i, i]:.2f}", ha="center", va="center", fontsize=6,
               color="white" if cm[i, i] > .5 else "black")
plt.tight_layout(); fig.savefig(DIR / "comparativa_confusiones.png", dpi=150); plt.show()

**DISCUSIÓN**

*(Completar con las cifras de las tablas anteriores tras reentrenar todos los modelos con el dataset actual.)*

**Resultados.** La CNN es el mejor modelo propio con diferencia en F1 macro; entre los clásicos, combinar descriptores (HOG + LBP) y reducir con PCA mejora a cada descriptor por separado, y las autocaras (píxeles + PCA) quedan por debajo porque modelan la apariencia global (iluminación, identidad) y no la forma local de la expresión. KNN es el clásico más débil en las clases difíciles. Todos coinciden en las clases fáciles (`happy`, `surprise`, `duckface`) y en las difíciles (`fear`, `sad`, `angry`).

**Facilidad de uso.**
- Clásicos: pocas líneas con scikit-learn y sin GPU, pero el preprocesado (HOG/LBP) hay que diseñarlo y ajustarlo a mano, y la SVM con kernel RBF escala mal con el número de muestras (entrenamiento aproximadamente cuadrático y modelos de cientos de MB porque guardan los vectores de soporte).
- CNN: más código e hiperparámetros (arquitectura, *learning rate*, aumento de datos, regularización), y entrenar en CPU tarda más de una hora, pero aprende las características sola y el modelo final ocupa ≈ 0,5 MB.
- DeepFace: una línea de código y sin entrenamiento, pero es una caja negra con clases fijas (no podríamos añadir `duckface`) y más lenta por cara.

**Rapidez de inferencia.** Para la webcam lo que importa es la latencia de una cara: la CNN con `tf.function` tarda pocos ms, los clásicos dependen del número de vectores de soporte (SVM) o de muestras guardadas (KNN), y DeepFace es el más lento, por lo que en el prototipo se ejecuta en un hilo aparte.

**Frente a DeepFace.** En la primera ejecución, sobre 600 caras equilibradas sin duckface, la CNN propia (0,570) y DeepFace (0,563) quedaron prácticamente empatadas en accuracy, y la CNN con TTA (0,603) lo superó, siendo además unas 6 veces más rápida por cara (≈ 1,5 ms frente a ≈ 9 ms). Hay que tener en cuenta que DeepFace se evalúa en su propio dominio (FER-2013).

**Conclusión.** Para el prototipo en tiempo real se elige la **CNN propia**: es la de mejor F1, la más ligera y suficientemente rápida, y además permite reconocer la clase propia `duckface`, cosa que DeepFace no puede hacer.